# SupplyIQ Exploratory Data Analysis
This notebook covers the EDA process for the Olist Dataset, fulfilling Module 2.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os

# Set style
sns.set_theme(style="whitegrid")
os.makedirs('../outputs', exist_ok=True)

# Connect to database
conn = sqlite3.connect('../outputs/supplyiq.db')

In [ ]:
## 1. Sales trend over time (line chart)
query = """
SELECT date(order_purchase_timestamp) as order_date, COUNT(order_id) as total_orders
FROM olist_orders_dataset
WHERE order_status != 'canceled'
GROUP BY order_date
ORDER BY order_date
"""
df_sales = pd.read_sql_query(query, conn)
df_sales['order_date'] = pd.to_datetime(df_sales['order_date'])

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_sales, x='order_date', y='total_orders')
plt.title('Daily Sales Volume Trend')
plt.xlabel('Date')
plt.ylabel('Total Orders')
plt.tight_layout()
plt.savefig('../outputs/sales_trend.png')
plt.show()

In [ ]:
## 2. Top product categories (bar chart)
query = """
SELECT p.product_category_name, COUNT(oi.order_id) as total_sold
FROM olist_order_items_dataset oi
JOIN olist_products_dataset p ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY total_sold DESC
LIMIT 10
"""
df_top_cat = pd.read_sql_query(query, conn)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_top_cat, y='product_category_name', x='total_sold', palette='viridis')
plt.title('Top 10 Product Categories by Units Sold')
plt.xlabel('Units Sold')
plt.ylabel('Category')
plt.tight_layout()
plt.savefig('../outputs/top_categories.png')
plt.show()

In [ ]:
## 3. Delivery delay heatmap by region
query = """
SELECT c.customer_state, 
       AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date)) as avg_delay
FROM olist_orders_dataset o
JOIN olist_customers_dataset c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered' 
AND o.order_delivered_customer_date > o.order_estimated_delivery_date
GROUP BY c.customer_state
"""
df_delay = pd.read_sql_query(query, conn)

plt.figure(figsize=(10, 6))
df_delay = df_delay.sort_values('avg_delay', ascending=False)
sns.barplot(data=df_delay, x='customer_state', y='avg_delay', palette='Reds_r')
plt.title('Average Delivery Delay Days by State')
plt.xlabel('State')
plt.ylabel('Average Delay (Days)')
plt.tight_layout()
plt.savefig('../outputs/delivery_delay_region.png')
plt.show()

In [ ]:
## 4. Order volume by day of week
query = """
SELECT strftime('%w', order_purchase_timestamp) as day_of_week, COUNT(order_id) as total_orders
FROM olist_orders_dataset
GROUP BY day_of_week
"""
df_dow = pd.read_sql_query(query, conn)
days = {0:'Sunday', 1:'Monday', 2:'Tuesday', 3:'Wednesday', 4:'Thursday', 5:'Friday', 6:'Saturday'}
df_dow['day_of_week'] = df_dow['day_of_week'].astype(int).map(days)

plt.figure(figsize=(8, 5))
sns.barplot(data=df_dow, x='day_of_week', y='total_orders', color='skyblue')
plt.title('Order Volume by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Total Orders')
plt.tight_layout()
plt.savefig('../outputs/order_volume_dow.png')
plt.show()

In [ ]:
## 5. Revenue distribution (histogram)
query = """
SELECT payment_value
FROM olist_order_payments_dataset
WHERE payment_value < 1000 -- filtering outliers for visualization
"""
df_rev = pd.read_sql_query(query, conn)

plt.figure(figsize=(10, 6))
sns.histplot(df_rev['payment_value'], bins=50, kde=True, color='purple')
plt.title('Distribution of Order Values (< $1000)')
plt.xlabel('Order Value ($)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('../outputs/revenue_distribution.png')
plt.show()